# Cell 1: Install & Imports

In [1]:
# Install the correct libraries for the Kaggle Python 3.12 environment
# !pip install transformers==4.35.0 tf-keras 
!pip install transformers==4.39.3 tf-keras xgboost scikit-learn
import os
import warnings
warnings.filterwarnings('ignore')

# Force the current notebook kernel to use Legacy Keras (just in case)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follo

Cell 2: Configuration (config.py)
This file controls your whole project. Change paths here only once.

In [2]:
%%writefile config.py
import os

# --- MASTER PATHS ---
BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/cfdf-preprocessed-dataset-non-splited/CelebDF_dataset_split" 
# BASE_DATA_PATH = "/kaggle/input/datasets/aashiqcse/ffpp-unsplited-alltogether/FFPP_dataset_split"
# BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/faceforensics-pngs/FFPP_Splitted_Dataset" 


TRAIN_PATH = os.path.join(BASE_DATA_PATH, "train")
VAL_PATH = os.path.join(BASE_DATA_PATH, "val")
TEST_PATH = os.path.join(BASE_DATA_PATH, "test")

# --- HYPERPARAMETERS ---
IMG_SIZE = (224, 224)
BATCH_SIZE_PER_REPLICA =32
EPOCHS = 20
LEARNING_RATE = 1e-4
DATA_SUBSET_RATIO = 0.99  

# --- OUTPUT FILES (UPDATED FOR 5 MODELS) ---
VIT_WEIGHTS_FILE = "/kaggle/input/models/artechie001/celeb-df-models-final/tensorflow2/default/1/vit_best.weights.h5"
XCP_ATTN_WEIGHTS_FILE = "xcp_attn_best.weights.h5"
XCP_BASE_WEIGHTS_FILE = "xcp_base_best.weights.h5"
EFFB5_WEIGHTS_FILE = "/kaggle/input/models/artechie001/celeb-df-models-final/tensorflow2/default/1/effb5_best.weights.h5"
DENSE_WEIGHTS_FILE = "dense201_best.weights.h5"



FEATURE_DIR = "Features_cfdf"

Writing config.py


# Cell 3: Utilities (utils.py)
Handles data loading and ImageNet normalization (required for ViT).

In [3]:
%%writefile utils.py
import config
import os
import glob
import tensorflow as tf
from sklearn.utils import shuffle

def get_image_paths(data_path, subset_ratio=config.DATA_SUBSET_RATIO):
    """
    Retrieves and shuffles image paths from the nested directory structure.
    """
    fake_paths = glob.glob(os.path.join(data_path, 'fake', '*', '*.*'))
    real_paths = glob.glob(os.path.join(data_path, 'real', '*', '*.*'))
    
    paths = fake_paths + real_paths
    labels = [1] * len(fake_paths) + [0] * len(real_paths) # 1 for Fake, 0 for Real
    
    paths, labels = shuffle(paths, labels, random_state=42)
    
    if subset_ratio < 1.0:
        limit = int(len(paths) * subset_ratio)
        paths = paths[:limit]
        labels = labels[:limit]
        
    print(f"Loaded {len(paths)} images from {data_path}")
    return paths, labels

def load_and_preprocess_image(path, label):
    """
    Native TensorFlow function to read, decode, resize, and normalize images.
    Crucial for preventing CPU bottlenecks during multi-GPU training.
    """

    img = tf.io.read_file(path)

    img = tf.image.decode_image(img, channels=3, expand_animations=False)

    img = tf.image.resize(img, config.IMG_SIZE)

    img = tf.cast(img, tf.float32) / 255.0
    

    label = tf.one_hot(label, depth=2)
    
    return img, label

def create_tf_dataset(paths, labels, batch_size, is_training=True):
    """
    Builds a highly optimized tf.data.Dataset pipeline.
    Uses AUTOTUNE to dynamically allocate CPU threads for data loading.
    """

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    if is_training:

        dataset = dataset.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
        

    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    

    dataset = dataset.batch(batch_size)
    

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

Writing utils.py


# Cell 4: Models (models.py)
Defines your ViXNet (ViT + Xception with Attention).

In [4]:
%%writefile models.py
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import Xception, EfficientNetB5, DenseNet201
from transformers import TFViTModel
from tensorflow.keras import mixed_precision

# Ensure final layers output in float32 for mixed precision stability
FINAL_DTYPE = 'float32'

class ViTWrapper(layers.Layer):
    """Custom layer to wrap the HuggingFace ViT model natively into Keras."""
    def __init__(self, vit_model, **kwargs):
        super().__init__(**kwargs)
        self.vit_model = vit_model
        
    def call(self, inputs):
        x = tf.transpose(inputs, perm=[0, 3, 1, 2])
        outputs = self.vit_model.vit(pixel_values=x)
        return outputs.last_hidden_state[:, 0, :]
        
    def get_config(self):
        return super().get_config()

class AttentionLayer(layers.Layer):
    """Custom Attention Mechanism for feature weighting."""
    def __init__(self, dim, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.dim = dim
        
    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim})
        return config
        
    def build(self, input_shape):
        self.dense = layers.Dense(self.dim, activation='tanh', use_bias=True)
        self.u_vec = self.add_weight(name='u_vec', shape=(self.dim, 1), initializer='uniform', trainable=True)
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        u_it = self.dense(x)
        score = tf.matmul(u_it, self.u_vec)
        weights = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

# 1. ViT
def build_vit_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    norm_layer = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])
    x = norm_layer(inputs)
    try:
        vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224', from_pt=True)
    except:
        vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224')
    vit_model.trainable = True
    features = ViTWrapper(vit_model, name='vit_features')(x)
    x = layers.Dense(512, activation='relu')(features)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="ViT_Classifier")

# 2. Xception + Custom Attention
def build_xception_attn_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs) 
    x = layers.GaussianNoise(0.05)(x)
    base = Xception(include_top=False, weights='imagenet', input_tensor=x)
    base.trainable = True
    x = base.output
    x = layers.Reshape((x.shape[1]*x.shape[2], x.shape[3]))(x)
    features = AttentionLayer(dim=512, name='xcp_features')(x)
    x = layers.Dense(512, activation='relu')(features)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="Xception_Attn_Classifier")

# 3. Base Xception
def build_xception_base_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs) 
    base = Xception(include_top=False, weights='imagenet', input_tensor=x, pooling='avg')
    base.trainable = True
    x = base.output
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="Xception_Base_Classifier")

# 4. EfficientNetB5
def build_effb5_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=255.0)(inputs) 
    
    # --- BULLETPROOF WORKAROUND FOR EFFICIENTNET ---
    # Temporarily drop back to float32 to bypass the internal hardcoded constants bug
    current_policy = mixed_precision.global_policy()
    mixed_precision.set_global_policy('float32')
    
    base = EfficientNetB5(include_top=False, weights='imagenet', pooling='avg')
    
    # Immediately restore the mixed precision policy
    mixed_precision.set_global_policy(current_policy)
    # -----------------------------------------------
    
    # Explicitly force input to float32 before feeding it to the base model
    x = tf.cast(x, tf.float32)
    x = base(x)
    
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="EfficientNetB5_Classifier")

# 5. DenseNet201
def build_dense201_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=255.0)(inputs) 
    
    # Apply standard preprocessing
    x = tf.cast(x, tf.float32)
    x = tf.keras.applications.densenet.preprocess_input(x) 
    
    # Apply the same safety wrapper for DenseNet
    current_policy = mixed_precision.global_policy()
    mixed_precision.set_global_policy('float32')
    
    base = DenseNet201(include_top=False, weights='imagenet', pooling='avg')
    
    mixed_precision.set_global_policy(current_policy)
    
    x = base(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="DenseNet201_Classifier")

Writing models.py


# 5. Feature_Extraction

In [5]:
%%writefile extract_features_separate.py
import config
import numpy as np
import os
import tensorflow as tf
from models import build_vit_classifier, build_effb5_classifier
from utils import get_image_paths, create_tf_dataset

def extract_combined_features(data_path, vit_extractor, eff_extractor, batch_size):
    paths, labels = get_image_paths(data_path)
    print(f"Extracting features from {len(paths)} images in {data_path}...")
    
    # Use the highly optimized dataset pipeline 
    # is_training=False ensures NO shuffling, so predictions perfectly align with our labels array
    dataset = create_tf_dataset(paths, labels, batch_size, is_training=False)
    
    # Predict in batches across GPUs 
    print("Extracting ViT global features (768-d)...")
    vit_features = vit_extractor.predict(dataset, verbose=1)
    
    print("Extracting EfficientNetB5 spatial features (2048-d)...")
    eff_features = eff_extractor.predict(dataset, verbose=1)
    
    # Manual Concatenation horizontally (768 + 2048 = 2816-d)
    print("Concatenating features...")
    combined_features = np.concatenate([vit_features, eff_features], axis=1)
    
    return combined_features, np.array(labels)

if __name__ == "__main__":
    print("--- PHASE 2: EXTRACTING HYBRID FEATURES (ViT + EfficientNetB5) ---")
    
    # Sync scope to distribute the heavy extraction workload across both T4 GPUs
    strategy = tf.distribute.MirroredStrategy()
    GLOBAL_BATCH_SIZE = config.BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

    with strategy.scope():
        # 1. Load ViT Model & Create Extractor
        print("\nLoading ViT Extractor...")
        vit_model = build_vit_classifier()
        vit_model.load_weights(config.VIT_WEIGHTS_FILE)
        # Target the custom 'vit_features' layer
        vit_extractor = tf.keras.Model(inputs=vit_model.input, outputs=vit_model.get_layer('vit_features').output)
        
        # 2. Load EfficientNetB5 Model & Create Extractor
        print("Loading EfficientNetB5 Extractor...")
        eff_model = build_effb5_classifier()
        eff_model.load_weights(config.EFFB5_WEIGHTS_FILE)
        
        # THE FIX: Intercept the tensor entering the 512-Dense layer (index -3).
        # This completely avoids the nested graph disconnection error.
        eff_extractor = tf.keras.Model(inputs=eff_model.input, outputs=eff_model.layers[-3].input)
        
    # Ensure save directory exists
    os.makedirs(config.FEATURE_DIR, exist_ok=True)
    
    # 3. Process Training Data
    print("\n--- Processing Training Data ---")
    X_train, y_train = extract_combined_features(config.TRAIN_PATH, vit_extractor, eff_extractor, GLOBAL_BATCH_SIZE)
    np.save(f"{config.FEATURE_DIR}/X_train.npy", X_train)
    np.save(f"{config.FEATURE_DIR}/y_train.npy", y_train)
    
    # 4. Process Testing Data
    print("\n--- Processing Testing Data ---")
    X_test, y_test = extract_combined_features(config.TEST_PATH, vit_extractor, eff_extractor, GLOBAL_BATCH_SIZE)
    np.save(f"{config.FEATURE_DIR}/X_test.npy", X_test)
    np.save(f"{config.FEATURE_DIR}/y_test.npy", y_test)
    
    print(f"\n✔ Success! 2816-d Hybrid Feature arrays saved to '{config.FEATURE_DIR}/'")


Writing extract_features_separate.py


In [6]:
# GPU
!python extract_features_separate.py

2026-05-17 14:19:01.167037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779027541.357648      53 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779027541.418326      53 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779027541.862638      53 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779027541.862723      53 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779027541.862727      53 computation_placer.cc:177] computation placer alr